# 🥉 Bronze Layer - Raw Data Ingestion

## Propósito
Ingesta cruda de las 5 fuentes de datos CSV en formato Delta Lake con metadata de auditoría.

## Arquitectura Medallion - Bronze
La capa Bronze contiene:
* **Datos crudos** sin transformaciones de negocio
* **Metadata de ingesta** (timestamp_ingestion, source_file, load_date)
* **Particionamiento** por load_date para performance
* **Formato Delta** para ACID transactions y time travel

## Tablas Bronze Creadas
1. `workspace.churn_bronze.raw_demo_asociados` - Datos demográficos
2. `workspace.churn_bronze.raw_retiro_aportes` - Retiros de aportes 2022
3. `workspace.churn_bronze.raw_plano_ahorros` - Productos de ahorro
4. `workspace.churn_bronze.raw_detallado_operaciones` - Operaciones transaccionales
5. `workspace.churn_bronze.raw_publiturno` - Interacciones por canal

---
**Layer:** Bronze (Raw)  
**Última actualización:** 2026-05-17  
**Modo:** Full Load (Batch)

In [0]:
%run ./00_config

In [0]:
# ================================================================================
# GENERIC INGESTION FUNCTION FOR BRONZE LAYER
# ================================================================================

def ingest_csv_to_bronze(
    table_key: str,
    config: dict,
    execution_mode: str = "full"
) -> None:
    """
    Función genérica para ingestar un archivo CSV a la capa Bronze.
    
    Args:
        table_key: Clave de la tabla en SOURCE_TABLES_CONFIG
        config: Diccionario de configuración de la tabla
        execution_mode: Modo de ejecución ('full' o 'incremental')
    
    Proceso:
        1. Leer CSV desde raw data path
        2. Validar schema esperado
        3. Agregar metadata de ingesta
        4. Escribir en tabla Delta particionada
        5. Ejecutar OPTIMIZE + ZORDER
        6. Loggear estadísticas
    """
    try:
        logger.info(f"\n{'='*80}")
        logger.info(f"Iniciando ingesta: {table_key}")
        logger.info(f"{'='*80}")
        
        # 1. Leer archivo CSV
        logger.info(f"📂 Leyendo archivo: {config['file_path']}")
        df_raw = spark.read \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .option("encoding", "UTF-8") \
            .csv(config['file_path'])
        
        initial_count = df_raw.count()
        logger.info(f"✓ Archivo leído: {initial_count:,} registros")
        
        # 2. Validar schema
        validate_schema(df_raw, config['expected_columns'], table_key)
        
        # 3. Agregar metadata de ingesta
        logger.info(f"📝 Agregando metadata de ingesta...")
        df_bronze = add_ingestion_metadata(df_raw, config['file_name'])
        
        # 4. Escribir en tabla Delta con particionamiento
        bronze_table = config['bronze_table']
        logger.info(f"💾 Escribiendo en tabla Delta: {bronze_table}")
        
        if execution_mode == "full":
            # Full load: sobreescribir tabla completa
            df_bronze.write \
                .format("delta") \
                .mode("overwrite") \
                .partitionBy(config['partition_by']) \
                .option("overwriteSchema", "true") \
                .option("delta.autoOptimize.optimizeWrite", "true") \
                .option("delta.autoOptimize.autoCompact", "true") \
                .saveAsTable(bronze_table)
            logger.info(f"✓ Tabla creada/actualizada (OVERWRITE): {bronze_table}")
        else:
            # Incremental load: append
            df_bronze.write \
                .format("delta") \
                .mode("append") \
                .option("delta.autoOptimize.optimizeWrite", "true") \
                .option("delta.autoOptimize.autoCompact", "true") \
                .saveAsTable(bronze_table)
            logger.info(f"✓ Datos agregados (APPEND): {bronze_table}")
        
        # 5. Ejecutar OPTIMIZE + ZORDER
        if get_execution_params()["run_optimization"]:
            logger.info(f"⚡ Ejecutando OPTIMIZE + ZORDER...")
            zorder_cols = OPTIMIZE_CONFIG['optimize_zorder_columns']['bronze']
            optimize_delta_table(bronze_table, zorder_cols)
        
        # 6. Loggear estadísticas finales
        df_final = spark.table(bronze_table)
        log_ingestion_stats(df_final, bronze_table, "BRONZE_INGESTION")
        
        logger.info(f"✅ Ingesta completada exitosamente: {table_key}")
        logger.info(f"{'='*80}\n")
        
    except Exception as e:
        logger.error(f"❌ Error en ingesta de {table_key}: {str(e)}")
        raise

print("✓ Función de ingesta genérica cargada: ingest_csv_to_bronze()")

In [0]:
# ================================================================================
# INGESTA 1: DATOS DEMOGRÁFICOS - demo_asociados.csv
# ================================================================================

print("\n" + "="*80)
print("🔄 INGESTA 1/5: Datos Demográficos")
print("="*80)

ingest_csv_to_bronze(
    table_key="demo_asociados",
    config=SOURCE_TABLES_CONFIG["demo_asociados"],
    execution_mode="full"
)

print("\n✅ Ingesta 1/5 completada: demo_asociados")

In [0]:
# ================================================================================
# INGESTA 2: RETIROS DE APORTES - Retiro_aportes_2022.csv
# ================================================================================

print("\n" + "="*80)
print("🔄 INGESTA 2/5: Retiros de Aportes 2022")
print("="*80)

ingest_csv_to_bronze(
    table_key="retiro_aportes",
    config=SOURCE_TABLES_CONFIG["retiro_aportes"],
    execution_mode="full"
)

print("\n✅ Ingesta 2/5 completada: retiro_aportes")

In [0]:
# ================================================================================
# INGESTA 3: PRODUCTOS DE AHORRO - Plano_ahorros.csv
# ================================================================================

print("\n" + "="*80)
print("🔄 INGESTA 3/5: Productos de Ahorro")
print("="*80)

ingest_csv_to_bronze(
    table_key="plano_ahorros",
    config=SOURCE_TABLES_CONFIG["plano_ahorros"],
    execution_mode="full"
)

print("\n✅ Ingesta 3/5 completada: plano_ahorros")

In [0]:
# ================================================================================
# INGESTA 4: OPERACIONES TRANSACCIONALES - detalladoperaciones.csv
# ================================================================================

print("\n" + "="*80)
print("🔄 INGESTA 4/5: Operaciones Transaccionales")
print("="*80)

ingest_csv_to_bronze(
    table_key="detallado_operaciones",
    config=SOURCE_TABLES_CONFIG["detallado_operaciones"],
    execution_mode="full"
)

print("\n✅ Ingesta 4/5 completada: detallado_operaciones")

In [0]:
# ================================================================================
# INGESTA 5: INTERACCIONES POR CANAL - publiturno.csv
# ================================================================================

print("\n" + "="*80)
print("🔄 INGESTA 5/5: Interacciones por Canal")
print("="*80)

ingest_csv_to_bronze(
    table_key="publiturno",
    config=SOURCE_TABLES_CONFIG["publiturno"],
    execution_mode="full"
)

print("\n✅ Ingesta 5/5 completada: publiturno")

In [0]:
# ================================================================================
# RESUMEN DE INGESTA BRONZE - VALIDACIÓN FINAL
# ================================================================================

print("\n" + "="*80)
print("📊 RESUMEN DE INGESTA - CAPA BRONZE")
print("="*80)

# Listar todas las tablas Bronze creadas
bronze_tables = [
    config['bronze_table'] 
    for config in SOURCE_TABLES_CONFIG.values()
]

print(f"\n✅ Total de tablas creadas: {len(bronze_tables)}\n")

# Crear resumen con conteos y metadata
summary_data = []

for table_name, config in SOURCE_TABLES_CONFIG.items():
    bronze_table = config['bronze_table']
    
    # Obtener estadísticas de la tabla
    df = spark.table(bronze_table)
    row_count = df.count()
    
    # Obtener fecha de carga más reciente
    latest_load = df.selectExpr("max(load_date) as max_date").collect()[0]['max_date']
    
    summary_data.append({
        "Tabla": table_name,
        "Tabla_Bronze": bronze_table,
        "Registros": row_count,
        "Ultima_Carga": str(latest_load),
        "Archivo_Origen": config['file_name']
    })

# Crear DataFrame de resumen
df_summary = spark.createDataFrame(summary_data)

print("Detalle de tablas Bronze:")
print("="*80)
display(df_summary)

# Estadísticas totales
total_records = sum([item['Registros'] for item in summary_data])

print("\n" + "="*80)
print("ESTADÍSTICAS TOTALES")
print("="*80)
print(f"Total de tablas:      {len(bronze_tables)}")
print(f"Total de registros:   {total_records:,}")
print(f"Esquema Bronze:       {BRONZE_SCHEMA}")
print(f"Formato:              Delta Lake")
print(f"Particionamiento:     load_date")
print(f"Optimización:         OPTIMIZE + ZORDER (Nit, load_date)")
print("="*80)

print("\n✅ INGESTA BRONZE COMPLETADA EXITOSAMENTE")
print("\n📍 Siguiente paso: Ejecutar notebook 02_silver_transformation")
print("   Comando: %run ./02_silver_transformation\n")

In [0]:
%sql
-- Verificar que todas las tablas Bronze fueron creadas correctamente
SHOW TABLES IN workspace.churn_bronze;

In [0]:
%sql
-- Muestra de datos de la tabla demo_asociados con metadata de ingesta
SELECT 
    Nit,
    Sexo,
    Fechnacimiento,
    Permanencia,
    Nivel_ingresos,
    Uso_creditos,
    source_file,
    timestamp_ingestion,
    load_date
FROM workspace.churn_bronze.raw_demo_asociados
LIMIT 10;